# 2 · Your first call to the model

Scratch space for **Milestone 2**. Each cell here calls the model through the gateway and spends a little of your budget - a few hundred tokens per run.

The second half shows a **tool call on the wire**, before a framework hides it. Read it once carefully: everything an agent does is this, in a loop.

In [ ]:
# Run this cell first, every time. It lets the notebook import the assignment's
# code, and re-imports your .py files automatically whenever you save them.
%load_ext autoreload
%autoreload 2

import os, sys
from pathlib import Path

ROOT = Path.cwd() if (Path.cwd() / "agent").is_dir() else Path.cwd().parent
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("working in", ROOT)

## One question, one answer

Finish `chat()` in `services/llm_client.py`, then:

In [ ]:
from services.llm_client import chat

print(chat("In one sentence: what is a prerequisite in a university curriculum?"))

## The model does not know your data

Ask it something only the database can answer.

In [ ]:
print(chat("What is the average percentage of the CSE department at CCIT for Batch 2024?"))

Whatever it said, it did not look anything up - it cannot. Either it told you so, or it produced a confident number from nowhere. That second behaviour is the problem this whole assignment exists to solve.

## A tool call, by hand

Now offer the model a *tool*: a function it may ask us to run. We describe it; the model decides whether to use it and with which arguments. It never runs anything itself.

In [ ]:
import json
from openai import OpenAI
from services.config import get_settings

settings = get_settings()
client = OpenAI(api_key=settings.api_key, base_url=settings.gateway_url)

tools = [{
    "type": "function",
    "function": {
        "name": "department_average",
        "description": "Return the average percentage of marks for one department of CCIT.",
        "parameters": {
            "type": "object",
            "properties": {"department": {"type": "string", "description": "Department code, e.g. CSE"}},
            "required": ["department"],
        },
    },
}]

messages = [{"role": "user", "content": "What is the average percentage of the CSE department?"}]
first = client.chat.completions.create(model=settings.model, messages=messages, tools=tools, max_tokens=1024)
reply = first.choices[0].message
print("finish_reason:", first.choices[0].finish_reason)
print("tool_calls   :", [(c.function.name, c.function.arguments) for c in (reply.tool_calls or [])])

`finish_reason` is `tool_calls`: the model did not answer - it asked. Now *we* run the function (a real query, if you have done Milestone 1) and send the result back as a `tool` message.

In [ ]:
from services.dataset_service import connect

def department_average(department: str) -> dict:
    with connect() as conn:
        row = conn.execute(
            "SELECT ROUND(AVG(marks * 100.0 / total_marks), 2) FROM student_performance WHERE department = %s",
            (department,)).fetchone()
    return {"department": department, "average_percentage": float(row[0]) if row[0] is not None else None}

call = reply.tool_calls[0]
result = department_average(**json.loads(call.function.arguments))
print("our function returned:", result)

messages += [reply.model_dump(exclude_none=True),
             {"role": "tool", "tool_call_id": call.id, "content": json.dumps(result)}]
second = client.chat.completions.create(model=settings.model, messages=messages, tools=tools, max_tokens=1024)
print(second.choices[0].message.content)

That is the whole mechanism:

1. send the conversation **and the tool descriptions**
2. the model answers, or asks for a tool
3. run the tool, append the result
4. repeat until it answers

In Milestone 6 the Strands Agents SDK runs this loop for you. You will write the tools and their descriptions - which, as you just saw, are the only thing the model knows about them.